# NLP – Tutorial 1 Lab: German Text Corpus
**Group 2 – German** · model `de_core_news_md`
**Members:** Blevis Allushi · Darsi Ramaj · Geri Belba

Run in **JupyterLab** (kernel with spaCy installed; see `README.md`). Order: **Kernel → Restart Kernel and Run All Cells**. Needs internet for the Wikipedia API and the one-time spaCy model download. If the setup cell had to install spaCy, restart the kernel once, then run all again.

| Part | Owner | Sections |
|---|---|---|
| Setup, pipeline demo, displaCy, Corpus collection, UTF-8 check | **Blevis** | 0, 1, 5 |
| Sources, licences, discussion, quality problems | **Geri** | 2, 4, 7 |
| Corpus statistics, quality problems, table row | **Darsi** | 3, 4, 6 |

## 0 · Setup and version printout  *(Blevis)*

In [ ]:
# Installs into the *kernel's* environment (skip these two lines if you already followed README.md)
%pip install -q -U spacy requests pandas
import sys
!{sys.executable} -m spacy download de_core_news_md

In [ ]:
import re, csv, spacy, pandas as pd
from pathlib import Path
from collections import Counter
from spacy import displacy
from IPython.display import display, HTML, Markdown

print('spaCy version:', spacy.__version__)
nlp = spacy.load('de_core_news_md')
print('Model:', nlp.meta['name'], nlp.meta['version'], '| pipeline:', nlp.pipe_names)

# One sentence through the full pipeline
doc = nlp('Johann Wolfgang von Goethe wurde 1749 in Frankfurt am Main geboren.')
display(pd.DataFrame([(t.i, t.text, t.lemma_, t.pos_, t.tag_, t.dep_, t.head.text, t.ent_type_) for t in doc],
                     columns=['i','token','lemma','pos','tag','dep','head','ent']))
print('Entities:', [(e.text, e.label_) for e in doc.ents])

In [ ]:
displacy.render(doc, style='dep', jupyter=True)
displacy.render(doc, style='ent', jupyter=True)

## 1 · Collecting the texts  *(Blevis)*
Text type: **introductions of German Wikipedia articles** (CC BY-SA 4.0, reuse allowed with attribution). Fetched through the official MediaWiki API (no scraping). Each intro is cut at a sentence boundary to at most 300 words; articles with fewer than 100 words are skipped. Metadata (title, URL, retrieval date, last-modified date, licence) is written to `sources.csv`.

In [ ]:
import requests, re, csv, datetime, time
from pathlib import Path

CORPUS = Path('corpus'); CORPUS.mkdir(exist_ok=True)
API = 'https://de.wikipedia.org/w/api.php'
HEADERS = {'User-Agent': 'UMT-NLP-Lab/1.0 (university course corpus; Wikipedia API, low volume)'}
LICENCE = 'CC BY-SA 4.0'
MAX_TEXTS, MIN_W, MAX_W = 30, 100, 300

# Mixed topics on purpose (cities, nature, people, companies) -> umlauts, ß, long compounds, many entities
TITLES = ['Berlin','München','Hamburg','Köln','Frankfurt am Main','Stuttgart','Düsseldorf','Leipzig','Dresden',
          'Nürnberg','Bremen','Hannover','Heidelberg','Rhein','Donau','Elbe','Bodensee','Schwarzwald','Zugspitze',
          'Harz','Johann Wolfgang von Goethe','Albert Einstein','Ludwig van Beethoven','Johann Sebastian Bach',
          'Immanuel Kant','Volkswagen','Siemens','Bundesrepublik Deutschland','Oktoberfest','Brandenburger Tor',
          'Rügen','Mosel (Fluss)','Lüneburger Heide','Weimar']

def fetch_intro(title):
    r = requests.get(API, headers=HEADERS, timeout=30, params=dict(
        action='query', format='json', prop='extracts|info', exintro=1, explaintext=1,
        redirects=1, inprop='url', titles=title))
    r.raise_for_status()
    return next(iter(r.json()['query']['pages'].values()))

def cut_to_words(text, max_w=MAX_W):
    text = re.sub(r'\s*\n+\s*', ' ', text).strip()
    out, n = [], 0
    for s in re.split(r'(?<=[.!?])\s+(?=[A-ZÄÖÜ])', text):
        w = len(s.split())
        if n + w > max_w: break
        out.append(s); n += w
    return ' '.join(out), n

rows, skipped = [], []
today = datetime.date.today().isoformat()
for title in TITLES:
    if len(rows) >= MAX_TEXTS: break
    try:
        p = fetch_intro(title)
        text, n = cut_to_words(p.get('extract', ''))
    except Exception as e:
        skipped.append((title, repr(e))); continue
    if n < MIN_W:
        skipped.append((title, f'only {n} words')); continue
    fname = f'text{len(rows)+1:02d}.txt'
    (CORPUS / fname).write_text(text, encoding='utf-8')
    rows.append(dict(file=fname, title=p['title'], url=p['fullurl'], retrieved=today,
                     last_modified=p.get('touched','')[:10], licence=LICENCE, words=n))
    time.sleep(0.3)

with open('sources.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
print(f'Saved {len(rows)} texts. Skipped: {skipped}')
pd.DataFrame(rows)

In [ ]:
# UTF-8 check: re-open files, look for mojibake and confirm special characters survive
CORPUS = Path('corpus')
MOJIBAKE = re.compile(r'Ã.|â€|\ufffd|Â')
bad = []
from collections import Counter
chars = Counter()
for f in sorted(CORPUS.glob('*.txt')):
    t = f.read_text(encoding='utf-8')
    if MOJIBAKE.search(t): bad.append(f.name)
    chars.update(c for c in t if c in 'äöüÄÖÜß')
print('Files with mojibake:', bad or 'none')
print('Special characters found:', dict(chars))
sample = next(s for f in sorted(CORPUS.glob('*.txt')) for s in f.read_text(encoding='utf-8').split('. ') if any(c in s for c in 'äöüß'))
print('Sample:', sample)

## 2 · Sources and licences  *(Geri)*

In [ ]:
src = pd.read_csv('sources.csv')
lines = [f'- **{r.title}** – <{r.url}> – retrieved {r.retrieved} (last modified {r.last_modified}) – {r.licence}' for r in src.itertuples()]
display(Markdown('All texts are introductions of German Wikipedia articles, licensed **CC BY-SA 4.0** (authors: see each article history).\n\n' + '\n'.join(lines)))

## 3 · Corpus statistics  *(Darsi)*

In [ ]:
from collections import Counter
files = sorted(Path('corpus').glob('*.txt'))
texts = [f.read_text(encoding='utf-8') for f in files]
tokens = []
for d in nlp.pipe(texts):
    tokens += [t.text.lower() for t in d if t.is_alpha]

print('Texts:', len(texts))
print('Tokens:', len(tokens))
print('Unique words:', len(set(tokens)))
top10 = Counter(tokens).most_common(10)
print(top10)
sw = sum(c for w, c in Counter(tokens).items() if w in nlp.Defaults.stop_words)
print(f'Stop-word share of all tokens: {sw/len(tokens):.1%}')

**Top-10 words.** *(Darsi: after the run, replace this with 2–3 sentences.)* Expected: nearly all are German function words (`der`, `die`, `und`, `in`, `von`, `das`, `den`, `ist`, `mit`, `im`…), i.e. stop words, which say almost nothing about the topic. Check how many of your ten are in `nlp.Defaults.stop_words` and which, if any, are content words (e.g. `deutschland`, `stadt`), since that reflects the topic mix.

## 4 · Quality problems  *(Geri & Darsi)*

In [ ]:
checks = {
  'HTML tags':            re.compile(r'<[^>]+>'),
  'HTML entities':        re.compile(r'&(?:[a-z]+|#\d+);'),
  'wiki markup (==, [[)': re.compile(r'==|\[\[|\]\]|\{\{'),
  'empty brackets ( ; )': re.compile(r'\(\s*[;,]?\s*\)'),
  'non-Latin script':     re.compile(r'[\u0370-\u03FF\u0400-\u04FF\u0590-\u06FF\u0900-\u0DFF\u3040-\u30FF\u4E00-\u9FFF]'),
  'IPA / pronunciation':  re.compile(r'[ɐ-˿]'),
  'boilerplate words':    re.compile(r'Cookie|Impressum|Navigation|Menü|Weiterleitung|Siehe auch|Einzelnachweise', re.I),
  'double spaces':        re.compile(r'  +'),
}
rep = {k: [] for k in checks}
for f, t in zip(files, texts):
    for k, rx in checks.items():
        m = rx.findall(t)
        if m: rep[k].append((f.name, len(m), m[0]))
for k, v in rep.items():
    print(f'{k:24s} files affected: {len(v)}', v[:3])

# repeated lines / repeated sentences across the corpus
sent_counts = Counter(s.strip() for t in texts for s in re.split(r'(?<=[.!?])\s+', t) if len(s.split()) > 4)
print('Repeated sentences:', [s for s, c in sent_counts.items() if c > 1][:5] or 'none')

# sentences that look English (very rough: >=2 common English function words)
EN = {'the','and','of','is','was','with','for','from','by','at'}
foreign = [(f.name, s) for f, t in zip(files, texts) for s in re.split(r'(?<=[.!?])\s+', t)
           if len(EN & set(re.findall(r'[a-z]+', s.lower()))) >= 2]
print('Possibly non-German sentences:', foreign[:5] or 'none')

*(Darsi: summarise what the cell found in 2–4 lines for the table's "Problems found" column. Typical for Wikipedia plaintext: empty brackets left after removed pronunciation/native-name templates, non-Latin script in name variants, no HTML.)*

## 5 · Three sentences through the pipeline  *(Blevis)*

In [ ]:
# Candidates picked automatically from the corpus; override SENTENCES with your own picks if you prefer.
pool = [s for d in nlp.pipe(texts) for s in d.sents if 8 <= len(s) <= 30]
A = max(pool, key=lambda s: len(s.ents))                                                    # most entities
B = max(pool, key=lambda s: max(len(t.text) for t in s if t.is_alpha))                      # longest word (compound)
CONTR = {'im','zum','zur','vom','beim','am','ins','ans','aufs','vom'}
C = next((s for s in pool if any(t.lower_ in CONTR for t in s) and s.text != A.text and s.text != B.text), pool[0])
SENTENCES = [A.text.strip(), B.text.strip(), C.text.strip()]
for i, s in enumerate(SENTENCES, 1): print(i, s, '\n')

In [ ]:
Path('figures').mkdir(exist_ok=True)
for i, s in enumerate(SENTENCES, 1):
    d = nlp(s)
    print(f'=== Sentence {i} ===\n{s}')
    display(pd.DataFrame([(t.text, t.lemma_, t.pos_, t.dep_, t.head.text, t.ent_type_) for t in d],
                         columns=['token','lemma','pos','dep','head','ent']))
    svg = displacy.render(d, style='dep', jupyter=False, options={'distance': 90})
    Path(f'figures/sentence{i}_dep.svg').write_text(svg, encoding='utf-8')
    display(HTML(svg))
    displacy.render(d, style='ent', jupyter=True)

### Sentence 1:
Düsseldorf ([ˈdʏsl̩ˌdɔʁf], Düsseldorfer Platt Düsseldörp) ist die Landeshauptstadt Nordrhein-Westfalens und der Behördensitz des Regierungsbezirks Düsseldorf.

Correct: the core structure is right. Düsseldorf is the subject (sb) of ist, Nordrhein-Westfalens is a genitive attribute (ag) of Landeshauptstadt, and und correctly links the two nouns as a coordination (cd/cj).

Incorrect: Behördensitz is tagged PROPN (proper noun) but is a common noun, so it should be NOUN. German capitalizes all nouns, which probably misleads the tagger. The IPA pronunciation is also treated as a normal token (tagged PROPN, labelled punct), although it is not German text.

### Sentence 2:
Im Zuge der Montagsdemonstrationen 1989, die einen entscheidenden Impuls für den Zusammenbruch der DDR gaben, wurde Leipzig als Heldenstadt bezeichnet.

Correct: the long-distance structure is handled well. The relative pronoun die is the subject of gaben, the relative clause (rc) attaches to Montagsdemonstrationen, and the passive wurde … bezeichnet is linked correctly (oc).

Incorrect: I can’t find a clear error in the POS tags or dependencies. The only debatable spot is als Heldenstadt labelled mo (modifier). That is arguably a complement of bezeichnet, but I’m not sure enough to call it wrong. Send me the lemma/entity table for this sentence and I’ll check there. You can also swap in a different sentence, which is legitimate as long as you only report errors the pipeline actually made.

### Sentence 3:
Neben den Flüssen Spree, Havel und Dahme befinden sich im Stadtgebiet kleinere Fließgewässer sowie zahlreiche Seen und Wälder.

Correct: the coordinations (Spree, Havel und Dahme, Fließgewässer sowie Seen und Wälder) are parsed correctly. The postposed subject Fließgewässer is attached to befinden, and sich is attached as its reflexive object.

Incorrect: the fronted phrase Neben den Flüssen … is attached to the noun Fließgewässer (mnr, postnominal modifier). It belongs to the verb befinden, because in German the phrase before the finite verb is part of the verb’s clause. A noun that comes after the verb cannot own it.

## 6 · German-specific behaviour and our table row  *(Darsi)*

In [ ]:
tests = ['Kraftfahrzeug-Haftpflichtversicherung', 'Donaudampfschifffahrtsgesellschaft', 'zum Bahnhof', 'im Haus', 'vom Rhein', 'Straße', 'Müller-Lüdenscheidt']
for s in tests:
    d = nlp(s)
    print(f'{s!r:42} ->', [(t.text, t.lemma_, t.pos_) for t in d])

In [ ]:
row = pd.DataFrame([dict(Language='German', Source='de.wikipedia.org (article introductions)', Licence='CC BY-SA 4.0',
                         Texts=len(texts), Tokens=len(tokens), **{'Unique words': len(set(tokens))},
                         **{'Problems found': 'FILL IN from section 4'})])
display(row)

**Discussion notes (German side)** *(Darsi: edit after looking at the output above)*
1. *Problems & causes:* fill from section 4.
2. *Long words / accents / contractions:* compounds stay one token, even hyphenated ones (`Kraftfahrzeug-Haftpflichtversicherung` came out as a single NOUN, with no decompounding); `zum`, `im`, `vom` also stay single tokens (no split into `zu dem`), but the lemmatizer maps them to `zu`/`in`; umlauts and ß are handled natively. Confirm with the cell above on your real corpus.
3. *What to change before real use:* fill in as a group, e.g. de-duplicate near-identical intros, strip bracket artefacts, balance topic types, keep per-text metadata.

## 7. Discussion (Geri)